In [13]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier, XGBRegressor

base = Path("/app")
dataset_dir = base / "ML_Training_datasets" / "Datasets" / "5S"
model_dir = base / "ML_Training_datasets" / "Models" / "5S_h10"
model_dir.mkdir(parents=True, exist_ok=True)

random_state = 42
target_col = "future_return_pct_10"


In [14]:
csv_files = sorted(dataset_dir.glob("memecoin_training_dataset_*.csv"))
print(f"Found {len(csv_files)} dataset files.")

dfs = []
for path in csv_files:
    df_part = pd.read_csv(path)
    df_part["source_file"] = path.name
    if "token_id" not in df_part.columns:
        token_stub = path.stem.replace("memecoin_training_dataset_", "token_")
        df_part["token_id"] = token_stub
    if "token_name" not in df_part.columns:
        df_part["token_name"] = df_part["token_id"]
    dfs.append(df_part)

if not dfs:
    raise ValueError(f"No dataset CSVs found in {dataset_dir}")

df = pd.concat(dfs, ignore_index=True)
print(df.shape)
df.head()


Found 19 dataset files.
(231033, 52)


,rsi,ema10,ema20,ema50,ema100,ema200,ema_cross,macd_line,macd_signal,macd_hist,...,future_return_pct_20,future_close_30,future_return_pct_30,future_close_40,future_return_pct_40,future_close_50,future_return_pct_50,token_id,token_name,source_file
0,76.64,3400727.99,3368076.84,3321586.72,3282056.03,3242820.99,1,38297.54,30072.09,8225.44,...,0.111953,3.479441e+06,1.460234,3.482963e+06,1.562949,3.423032e+06,-0.184629,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,memecoin_training_dataset_1.csv
1,74.91,3405330.96,3373597.55,3325683.09,3284907.28,3244644.10,1,38102.21,31678.12,6424.10,...,0.033722,3.478758e+06,1.538611,3.426884e+06,0.024518,3.422471e+06,-0.104287,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,memecoin_training_dataset_1.csv
2,77.33,3411744.54,3379979.28,3330189.86,3287990.42,3246593.97,1,38676.55,33077.80,5598.75,...,0.034271,3.473882e+06,0.967165,3.432200e+06,-0.244299,3.421427e+06,-0.557407,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,memecoin_training_dataset_1.csv
3,71.02,3414725.35,3384565.92,3334031.00,3290765.64,3248400.39,1,37691.29,34000.50,3690.79,...,-0.096426,3.479687e+06,1.503666,3.425412e+06,-0.079553,3.418651e+06,-0.276783,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,memecoin_training_dataset_1.csv
4,71.13,3417259.39,3388765.60,3337742.05,3293496.27,3250194.04,1,36531.59,34506.72,2024.87,...,1.461377,3.473643e+06,1.311900,3.433468e+06,0.140156,3.423067e+06,-0.163203,2JBXEbEfzMNZGmP8fRhsUjJQqZUDLhJzwBjvh7vBw7Df,Jellybean,memecoin_training_dataset_1.csv


In [15]:
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df = df.sort_values(["token_id", "timestamp"]).reset_index(drop=True)

le_supertrend = LabelEncoder()
df["supertrend_trend_enc"] = le_supertrend.fit_transform(df["supertrend_trend"].astype(str))

def add_group_features(g):
    g = g.copy()
    g["rsi_lag_1"] = g["rsi"].shift(1)
    g["rsi_roll_3"] = g["rsi"].rolling(3, min_periods=1).mean()
    g["rsi_roll_5"] = g["rsi"].rolling(5, min_periods=1).mean()
    g["atr_roll_3"] = g["atr"].rolling(3, min_periods=1).mean()
    g["atr_roll_5"] = g["atr"].rolling(5, min_periods=1).mean()
    g["volume_avg_roll_3"] = g["volume_avg"].rolling(3, min_periods=1).mean()
    g["volume_avg_roll_5"] = g["volume_avg"].rolling(5, min_periods=1).mean()
    g["obv_roll_3"] = g["obv"].rolling(3, min_periods=1).mean()
    return g

token_id_backup = df["token_id"].copy()
df = df.groupby("token_id", group_keys=False).apply(add_group_features)
if "token_id" not in df.columns:
    df["token_id"] = token_id_backup.loc[df.index].to_numpy()

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna().reset_index(drop=True)
print(df.shape)


(231014, 61)


In [16]:
base_feature_cols = [
    "rsi", "ema10", "ema20", "ema50", "ema100", "ema200", "ema_cross",
    "macd_line", "macd_signal", "macd_hist", "vwap", "adx", "plus_di",
    "minus_di", "stoch_k", "stoch_d", "boll_upper", "boll_middle",
    "boll_lower", "boll_percent", "atr", "obv", "supertrend_value",
    "cci", "roc", "momentum3", "volume_sum", "volume_avg", "range_pct",
    "open_rel", "high_rel", "low_rel", "close_rel", "hour", "minute",
    "weekday", "supertrend_trend_enc", "rsi_lag_1",
    "rsi_roll_3", "rsi_roll_5", "atr_roll_3", "atr_roll_5",
    "volume_avg_roll_3", "volume_avg_roll_5", "obv_roll_3"
]

optional_feature_cols = [
    "last_5m_buyCount", "last_5m_sellCount", "last_5m_buyVolumeSol",
    "last_5m_sellVolumeSol", "last_5m_priceSol"
]

feature_cols = base_feature_cols + [c for c in optional_feature_cols if c in df.columns]
missing_required = [c for c in base_feature_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required feature columns: {missing_required}")

X = df[feature_cols].copy()
tokens = np.array(sorted(df["token_id"].unique()))
print(f"Rows: {len(df)}, features: {len(feature_cols)}, tokens: {len(tokens)}")


Rows: 231014, features: 45, tokens: 19


In [17]:
train_tokens, test_tokens = train_test_split(tokens, test_size=0.2, random_state=random_state)
train_tokens, val_tokens = train_test_split(train_tokens, test_size=0.2, random_state=random_state)

train_mask = df["token_id"].isin(train_tokens)
val_mask = df["token_id"].isin(val_tokens)
test_mask = df["token_id"].isin(test_tokens)

split_summary = pd.DataFrame([
    {"split": "train", "tokens": len(train_tokens), "rows": int(train_mask.sum())},
    {"split": "val", "tokens": len(val_tokens), "rows": int(val_mask.sum())},
    {"split": "test", "tokens": len(test_tokens), "rows": int(test_mask.sum())},
])
split_summary


,split,tokens,rows
0,train,12,159895
1,val,3,34947
2,test,4,36172


In [18]:
X_train = X.loc[train_mask]
X_val = X.loc[val_mask]
X_test = X.loc[test_mask]

y_train = df.loc[train_mask, target_col]
y_val = df.loc[val_mask, target_col]
y_test = df.loc[test_mask, target_col]

clip_low, clip_high = np.percentile(y_train, [1, 99])
y_train_clipped = y_train.clip(clip_low, clip_high)
y_val_clipped = y_val.clip(clip_low, clip_high)
y_test_clipped = y_test.clip(clip_low, clip_high)

def regression_metrics(y_true, y_pred):
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "r2": r2_score(y_true, y_pred),
        "direction_acc": float((np.sign(y_true) == np.sign(y_pred)).mean()),
    }

baseline_zero_val = np.zeros(len(y_val))
baseline_zero_test = np.zeros(len(y_test))
baseline_mean = float(y_train.mean())
baseline_mean_val = np.full(len(y_val), baseline_mean)
baseline_mean_test = np.full(len(y_test), baseline_mean)

baseline_metrics = pd.DataFrame([
    {"model": "zero_baseline", "split": "val", **regression_metrics(y_val, baseline_zero_val)},
    {"model": "zero_baseline", "split": "test", **regression_metrics(y_test, baseline_zero_test)},
    {"model": "mean_baseline", "split": "val", **regression_metrics(y_val, baseline_mean_val)},
    {"model": "mean_baseline", "split": "test", **regression_metrics(y_test, baseline_mean_test)},
])
baseline_metrics


,model,split,mae,rmse,r2,direction_acc
0,zero_baseline,val,2.368653,3.713268,-0.000038,0.000000
1,zero_baseline,test,3.360348,6.305516,-0.000763,0.000000
2,mean_baseline,val,2.371437,3.713806,-0.000328,0.495608
3,mean_baseline,test,3.363359,6.303670,-0.000178,0.489882


In [19]:
reg_model = XGBRegressor(
    n_estimators=600,
    max_depth=5,
    learning_rate=0.04,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=random_state,
    objective="reg:squarederror",
    tree_method="hist",
)

reg_model.fit(
    X_train,
    y_train_clipped,
    eval_set=[(X_val, y_val_clipped)],
    verbose=False,
)

reg_val_pred = reg_model.predict(X_val)
reg_test_pred = reg_model.predict(X_test)

regression_results = pd.DataFrame([
    {"model": "xgb_reg_clipped", "split": "val", **regression_metrics(y_val, reg_val_pred)},
    {"model": "xgb_reg_clipped", "split": "test", **regression_metrics(y_test, reg_test_pred)},
])
regression_results


,model,split,mae,rmse,r2,direction_acc
0,xgb_reg_clipped,val,2.555084,3.945458,-0.129012,0.507225
1,xgb_reg_clipped,test,3.527105,6.562474,-0.083990,0.516836


In [20]:
def direction_label(series):
    labels = np.where(series > 0.0, "up", "down_or_flat")
    return pd.Series(labels, index=series.index)

y_train_dir = direction_label(y_train)
y_val_dir = direction_label(y_val)
y_test_dir = direction_label(y_test)

dir_encoder = LabelEncoder()
y_train_dir_enc = dir_encoder.fit_transform(y_train_dir)
y_val_dir_enc = dir_encoder.transform(y_val_dir)
y_test_dir_enc = dir_encoder.transform(y_test_dir)

class_counts = y_train_dir.value_counts().sort_index()
class_weight_scale = len(y_train_dir) / (len(class_counts) * class_counts)
sample_weight = y_train_dir.map(class_weight_scale).to_numpy()

clf_model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=random_state,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
)

clf_model.fit(
    X_train,
    y_train_dir_enc,
    sample_weight=sample_weight,
    eval_set=[(X_val, y_val_dir_enc)],
    verbose=False,
)

val_pred_dir = clf_model.predict(X_val)
test_pred_dir = clf_model.predict(X_test)

classification_metrics = pd.DataFrame([
    {"split": "val", "accuracy": accuracy_score(y_val_dir_enc, val_pred_dir)},
    {"split": "test", "accuracy": accuracy_score(y_test_dir_enc, test_pred_dir)},
])

classification_metrics


,split,accuracy
0,val,0.516754
1,test,0.522504


In [21]:
print("Validation classification report")
print(classification_report(y_val_dir_enc, val_pred_dir, target_names=dir_encoder.classes_))

print("Test classification report")
print(classification_report(y_test_dir_enc, test_pred_dir, target_names=dir_encoder.classes_))


Validation classification report
              precision    recall  f1-score   support

down_or_flat       0.52      0.53      0.53     17627
          up       0.51      0.50      0.51     17320

    accuracy                           0.52     34947
   macro avg       0.52      0.52      0.52     34947
weighted avg       0.52      0.52      0.52     34947

Test classification report
              precision    recall  f1-score   support

down_or_flat       0.53      0.55      0.54     18452
          up       0.51      0.49      0.50     17720

    accuracy                           0.52     36172
   macro avg       0.52      0.52      0.52     36172
weighted avg       0.52      0.52      0.52     36172



In [22]:
feature_importance = pd.Series(reg_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

joblib.dump(reg_model, model_dir / "xgb_reg_h10_clipped.joblib")
joblib.dump(clf_model, model_dir / "xgb_clf_h10_direction.joblib")
joblib.dump({
    "feature_cols": feature_cols,
    "target_col": target_col,
    "regression_clip": [float(clip_low), float(clip_high)],
    "supertrend_classes": list(le_supertrend.classes_),
    "direction_classes": list(dir_encoder.classes_),
    "train_tokens": list(train_tokens),
    "val_tokens": list(val_tokens),
    "test_tokens": list(test_tokens),
}, model_dir / "artifacts_h10.joblib")

baseline_metrics.to_csv(model_dir / "baseline_metrics_h10.csv", index=False)
regression_results.to_csv(model_dir / "regression_metrics_h10.csv", index=False)
classification_metrics.to_csv(model_dir / "classification_metrics_h10.csv", index=False)

feature_importance.head(20)


volume_avg_roll_3    0.035042
hour                 0.031127
volume_avg_roll_5    0.029399
ema50                0.029003
volume_sum           0.028247
ema200               0.027097
supertrend_value     0.026282
ema100               0.025718
boll_upper           0.025006
obv_roll_3           0.024932
vwap                 0.024583
boll_middle          0.024368
boll_lower           0.024075
low_rel              0.023936
minute               0.023733
macd_signal          0.023390
ema_cross            0.023264
open_rel             0.023212
range_pct            0.022926
atr_roll_5           0.022833
dtype: float32